# pipeline.ipynb
**Does:** generates the 13 µm/px synthetic sequences and runs every pipeline step on them (Steps 1-10), scoring each stage against the ground truth.
**Uses:** `src/` config, library, patterns, render, preprocess, segment, track, tables, classify, hitl, carbon, flux, evaluate, figures.
**Outputs:** `run_13um/`, `run_13um_rare/` and `pipeline_outputs/<n>/` (seg, tracks, tables, cls, eval, figures, evaluation_vs_truth.xlsx) in `MyDrive/snocam_pipeline/`.

## Settings

In [ ]:
# settings for the whole run (the only cell you normally edit)
UM_PER_PX     = 13.0        # SnoCam+ pixel size
KEEP_FOV      = True        # keep the 55.2 x 41.4 mm field of view
MIN_AREA_MODE = 'pixels'    # smallest counted particle stays 50 px (104 um)
CANVAS        = None        # classifier crop size in px (None = 80 px at 13 um/px)

N_SEQ, FRAMES, SPLIT_PLAN, SEED = 20, 30, '16,2,2', 0   # 16 train / 2 val / 2 test
N_RARE, RARE_SEED = 10, 7                                # extra training sequences, every class equally common
RUN_ID = None               # None = new output folder; a number = continue that run

TAU = 0.3                   # quality-filter cut-off of hybrid_segment
TRAIN_SEQS = ['seq000', 'seq003', 'seq006', 'seq009', 'seq012', 'seq015']   # filter training sequences
TRAIN_FRAME_STEP = 2
FILTER_FILE = None          # a saved filter to reuse, or None to train one

WIDE = True                 # widening search window (False = Song's fixed window)
EXCLUDE_EDGE = False        # drop arrivals flagged -7 (may have come in from the masked edge)

SEEDS = (0, 1, 2)           # ResNet-18 ensemble
OOD_EPOCHS, FT_EPOCHS = 30, 8
IMAGENET_SIZE = 128
CNN_MAX_PER_EPOCH = 40000
EVERY_TRAIN = 3             # every 3rd frame of the training sequences is used
BUDGETS = (25, 50, 100)     # particles per class checked by the expert
MINBOOST, RARE_SHOWN = 30, 30
TARGET_WEIGHT = 3.0
LAMBDA, K, REJECT = 0.25, 3, 0.5   # prior weight, frames per particle, false-detection threshold

FLUX_BUDGET = 50            # budget used for the flux results
CARBON_SETTING = 'auto'     # 'auto' = chosen on validation, or e.g. dict(size='measured', gate=True)

## Setup

In [ ]:
# mount Drive, load src/, set the pixel scale and create this run's output folder
import os, sys, glob, json, shutil, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from IPython.display import display
pd.set_option('display.max_columns', None, 'display.width', 250)

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/snocam_pipeline'
sh = get_ipython().system
sh('pip install -q timm openpyxl')
shutil.copytree(f'{DRIVE}/src', '/content/src', dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
sys.path.insert(0, '/content')
from src import config as C, render, preprocess, segment as sg, track as tk, tables as tb
from src import classify as cl, hitl, flux as fx, evaluate as ev, figures as fg

print(C.configure(UM_PER_PX, KEEP_FOV, MIN_AREA_MODE, CANVAS))
DATA_NAME = C.data_name()
RARE_NAME = DATA_NAME + '_rare'
DATA, RARE = f'/content/{DATA_NAME}', f'/content/{RARE_NAME}'

os.makedirs(f'{DRIVE}/pipeline_outputs', exist_ok=True)
if RUN_ID is None:
    used = [int(n) for n in os.listdir(f'{DRIVE}/pipeline_outputs') if n.isdigit()]
    RUN_ID = max(used) + 1 if used else 0
OUT = f'{DRIVE}/pipeline_outputs/{RUN_ID}'
EVAL, FIGS = f'{OUT}/eval', f'{OUT}/figures'
for d in (OUT, EVAL, FIGS):
    os.makedirs(d, exist_ok=True)
SETTINGS = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, tuple, dict, type(None)))}
json.dump(dict(SETTINGS, resolution=C.summary()), open(f'{OUT}/settings.json', 'w'), indent=1, default=str)
print('data:', DATA_NAME, '| outputs:', OUT)


def restore(name):
    # copy a dataset from Drive to /content (from its .tar if there is one)
    if os.path.exists(f'/content/{name}'):
        return True
    if os.path.exists(f'{DRIVE}/{name}.tar'):
        sh(f'tar -xf {DRIVE}/{name}.tar -C /content')
    elif os.path.exists(f'{DRIVE}/{name}'):
        shutil.copytree(f'{DRIVE}/{name}', f'/content/{name}')
    return os.path.exists(f'/content/{name}')


def backup(name, seq_dir):
    # copy one finished sequence to Drive
    dst = f'{DRIVE}/{name}/{os.path.basename(seq_dir)}'
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(seq_dir, dst)


def snapshot(name):
    # archive a complete dataset to Drive as one .tar (much faster to restore)
    sh(f'tar -cf /content/{name}.tar -C /content {name} && mv /content/{name}.tar {DRIVE}/{name}.tar')


def seqs(root, splits=('train', 'val', 'test')):
    return [d for d in sorted(glob.glob(f'{root}/*_*')) if os.path.isdir(d) and d.split('_')[-1] in splits]

## Status

In [ ]:
# restore the data and show what is already done
for name in ('lib', DATA_NAME, RARE_NAME):
    restore(name)
for name, root, n in ((DATA_NAME, DATA, N_SEQ), (RARE_NAME, RARE, N_RARE)):
    ss = seqs(root)
    print(f'{name}: {sum(os.path.exists(f"{d}/truth/truth.xlsx") for d in ss)}/{n} rendered, '
          f'{sum(preprocess.is_done(d) for d in ss)}/{n} background-removed')
print('filter:', os.path.exists(f'{OUT}/quality_filter.joblib'), '| segmented:', len(glob.glob(f'{OUT}/seg/*/detections.csv')),
      '| tracked:', len(glob.glob(f'{OUT}/tracks/*_counts.csv')), '| tables:', len(glob.glob(f'{OUT}/tables/*.pkl')),
      '| OOD models:', len(glob.glob(f'{OUT}/cls/models/OOD_s*.pt')))
try:
    import torch
    print('GPU:', torch.cuda.is_available())
except ImportError:
    print('GPU: torch not installed')

## Part A: synthetic data (Steps 1-5)

In [ ]:
# Steps 2-3: sprite library and scene patterns (from lib.tar, or built from Durkin's oblique crops)
LIB = '/content/lib'
if not restore('lib'):
    CROPS = '/content/data/All_obl_geltrap_particle'
    if not os.path.isdir(CROPS):
        sh('mkdir -p /content/data && cd /content/data && '
           'wget -q https://datadocs.bco-dmo.org/data/305/EAGER_DNA_BioPump/860725/1/data/Durkin_geltrap_oblique_particles.tar.gz && '
           'tar -xzf Durkin_geltrap_oblique_particles.tar.gz')
    sh(f'cd /content && python -m src.library {CROPS} {LIB} --group-by micrograph --max-per-class 2000')
    sh(f'cd /content && python -m src.patterns {LIB} --crops-dir {CROPS} --classes aggregate dense_detritus '
       'large_loose_pellet long_fecal_pellet short_pellet mini_pellet salp_pellet rhizaria phytoplankton fiber unidentifiable')
    snapshot('lib')
print(pd.read_csv(f'{LIB}/library.csv').groupby(['class', 'split']).size().unstack(fill_value=0))

In [ ]:
# Steps 4-5: render the main sequences, remove the background and back each one up to Drive
def finish(name):
    def f(seq_dir):
        preprocess.run(seq_dir)
        backup(name, seq_dir)
    return f


man = render.render_dataset(LIB, DATA, N_SEQ, FRAMES, SPLIT_PLAN, SEED, after_each=finish(DATA_NAME))
for d in seqs(DATA):
    if not preprocess.is_done(d):
        finish(DATA_NAME)(d)
if not os.path.exists(f'{DRIVE}/{DATA_NAME}.tar') and len(seqs(DATA)) == N_SEQ:
    snapshot(DATA_NAME)
display(man)

In [ ]:
# rare-enriched training sequences (every class equally common, extra swimmers)
man_r = render.render_dataset(LIB, RARE, N_RARE, FRAMES, '1,0,0', RARE_SEED, equal_classes=True, swimmers=(0.8, 1.2),
                              prefix='rare', after_each=finish(RARE_NAME))
for d in seqs(RARE):
    if not preprocess.is_done(d):
        finish(RARE_NAME)(d)
if not os.path.exists(f'{DRIVE}/{RARE_NAME}.tar') and len(seqs(RARE)) == N_RARE:
    snapshot(RARE_NAME)
display(man_r)

In [ ]:
# check the generator against the truth: frame size, sizes kept in um, arrivals
gen = ev.check_generation(seqs(DATA) + seqs(RARE))
display(gen)
gen.to_csv(f'{EVAL}/stage0_generator.csv', index=False)
TEST_SEQ = seqs(DATA, ('test',))[-1]
fg.generation(TEST_SEQ, FRAMES - 1, f'{FIGS}/stage0_generator.png'); plt.show()

In [ ]:
# check background removal: leftover background vs particle brightness, share of particles above T_seq
pre = pd.concat([ev.check_preprocessing(d, range(0, FRAMES, 5)) for d in seqs(DATA, ('val', 'test'))], ignore_index=True)
display(pre.groupby('seq_id')[['bg_p90', 'particle_mean_median', 'particles_above_T_seq_pct',
                                'contrast_particle_over_bg_p90']].mean().round(2))
pre.to_csv(f'{EVAL}/stage1_preprocessing.csv', index=False)

## Part B: segmentation (Step 6)

In [ ]:
# train the hybrid_segment quality filter (or load it) and set the cut-off TAU
import joblib
saved = f'{OUT}/quality_filter.joblib'
if FILTER_FILE:
    filt = joblib.load(FILTER_FILE)
elif os.path.exists(saved):
    filt = joblib.load(saved)
else:
    train = [d for d in seqs(DATA, ('train',)) if os.path.basename(d).split('_')[0] in TRAIN_SEQS]
    filt = sg.train_filter(train, TRAIN_FRAME_STEP)
    joblib.dump(filt, saved)
    filt['curve'].to_csv(f'{EVAL}/stage2_tau_curve.csv', index=False)
filt = sg.retune(filt, TAU)
print('tau =', filt['tau'], '| size correction:', filt.get('scale'))
display(filt['curve'])
fg.tau_curve(filt['curve'], filt['tau'], f'{FIGS}/stage2_tau_curve.png'); plt.show()

In [ ]:
# segment every val/test frame and every EVERY_TRAIN-th training frame
for d in seqs(DATA, ('val', 'test')):
    sg.segment_sequence(d, OUT, filt, 'all')
for d in seqs(DATA, ('train',)) + seqs(RARE):
    sg.segment_sequence(d, OUT, filt, f'every{EVERY_TRAIN}')

In [ ]:
# score segmentation against the truth (IoU >= 0.5)
VT = [os.path.basename(d) for d in seqs(DATA, ('val', 'test'))]
seg_frame, seg_sum = sg.summarise(OUT, VT)
display(seg_sum)
seg_sum.to_csv(f'{EVAL}/stage2_segmentation.csv', index=False)
seg_frame.to_csv(f'{EVAL}/stage2_segmentation_per_frame.csv', index=False)
fg.carbon_per_frame(seg_frame, f'{FIGS}/stage2_carbon_per_frame.png'); plt.show()
fg.seg_boxes(seqs(DATA, ('val',))[0], OUT, FRAMES - 1, f'{FIGS}/stage2_boxes.png'); plt.show()

## Part C: tracking (Step 7)

In [ ]:
# track every val/test sequence and score links and arrivals against the truth
for d in seqs(DATA, ('val', 'test')):
    tk.track_sequence(d, OUT, WIDE)
counts = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{OUT}/tracks/*_counts.csv')) if os.path.getsize(f) > 2])
trk = pd.concat([tk.summarise(counts, 'seq_id'), tk.summarise(counts, 'split'), tk.summarise(counts, None)])
display(trk.T)
trk.to_csv(f'{EVAL}/stage3_tracking.csv')

In [ ]:
# trajectories of the longest tracks (Song Fig. 3)
TT = pd.read_csv(f'{OUT}/tracks/{os.path.basename(TEST_SEQ)}.csv')
fg.trajectories(TEST_SEQ, TT, 6, f'{FIGS}/song_fig3_trajectories.png'); plt.show()

## Part D: particle tables (Step 8)

In [ ]:
# measurements, crop and truth of every detection; particles per class and size check
for d in seqs(DATA) + seqs(RARE):
    tb.build_sequence(d, OUT)
T_all = cl.add_labels(tb.load_tables(OUT, crops=False)[0])
p = T_all.drop_duplicates('pkey')
display(p.groupby(['label', 'split']).size().unstack(fill_value=0))
m = p[p.uid > 0]
print('false detections %:', T_all.groupby('split').uid.apply(lambda u: round(100 * (u == 0).mean(), 1)).to_dict())
print('median measured / true ESD:', (m.esd_um / m.esd_true).groupby(m.split).median().round(2).to_dict())
fg.class_distribution(T_all, f'{FIGS}/amaral_fig3_class_distribution.png'); plt.show()

## Part E: classification (Step 9, GPU)

In [ ]:
# train the out-of-domain ResNet-18 ensemble
H = hitl.Hitl(OUT, scheme='amaral7u', seeds=SEEDS, ood_epochs=OOD_EPOCHS, ft_epochs=FT_EPOCHS, input_size=IMAGENET_SIZE,
              max_per_epoch=CNN_MAX_PER_EPOCH, budgets=BUDGETS, minboost=MINBOOST, rare_shown=RARE_SHOWN,
              target_weight=TARGET_WEIGHT, lam=LAMBDA, K=K, reject=REJECT)
print('classes:', [str(c) for c in H.classes], '| training detections:', len(H.tr))
H.train_ood()

In [ ]:
# expert checks for every budget: +verify, then +rareboost
H.run_budgets()

In [ ]:
# score classification on the particles the expert never saw
R, recalls = H.results()
display(R[['domain', 'budget', 'stage', 'checked_pct', 'accuracy', 'balanced_accuracy', 'gelcam5_accuracy',
           'gelcam5_balanced_accuracy', 'accuracy_esd_ge_500um', 'n_particles']].round(3))
display((recalls * 100).round(1))
R.to_csv(f'{EVAL}/stage4_classification.csv', index=False)
recalls.to_csv(f'{EVAL}/stage4_recall_per_class.csv')
fg.effort_curve(R, f'{FIGS}/stage4_effort_curve.png'); plt.show()

FINAL = f'b{FLUX_BUDGET}_+rareboost'
STAGES = ['OOD', f'b{FLUX_BUDGET}_+verify', FINAL]
pr = {}
for dom in H.s['domains']:
    common = H.common_unseen(dom)
    for key, b, stage, checked in H.stages(dom):
        if key in STAGES:
            info = H.stage_particles(dom, key, checked)
            pr[f'{dom} {key}'] = cl.precision_recall(info, info.pred.values, common)
            if dom == 'test' and key in ('OOD', FINAL):
                cm = cl.confusion(info, info.pred.values, exclude=common)
                fg.confusion(cm, f'{FIGS}/stage4_confusion_{dom}_{key}.png', f'{dom}, {key}'); plt.show()
pd.concat(pr, names=['stage']).to_csv(f'{EVAL}/stage4_precision_recall.csv')

## Part F: carbon and flux (Step 10)

In [ ]:
# choose the size estimate for carbon on validation, then carbon errors of every stage
if CARBON_SETTING == 'auto':
    CS, grid = H.choose_carbon_setting(FLUX_BUDGET, 'val')
    display(grid.round(3))
else:
    CS = dict(CARBON_SETTING)
print('carbon setting:', CS)
json.dump(CS, open(f'{EVAL}/carbon_setting.json', 'w'))
R2, _ = H.results(CS['size'], CS['gate'])
display(R2[['domain', 'budget', 'stage', 'checked_pct', 'carbon_error_found', 'carbon_error_all']].round(3))
R2.to_csv(f'{EVAL}/stage5_carbon_errors.csv', index=False)

In [ ]:
# flux per sequence and per class vs the truth, cumulative carbon and the carbon budget
seq_vt = seqs(DATA, ('val', 'test'))
PARTS, FLUX_SEQ, FLUX_CLS = {}, {}, {}
for key in STAGES:
    parts = [H.stage_particles(dom, key, st[0][3], CS['size'], CS['gate'])
             for dom in H.s['domains'] for st in [[x for x in H.stages(dom) if x[0] == key]] if st]
    if parts:
        P = PARTS[key] = pd.concat(parts)
        FLUX_SEQ[key] = fx.flux_table(P, H.truth_arr, seq_vt, exclude_edge=EXCLUDE_EDGE)
        FLUX_CLS[key] = fx.flux_table(P, H.truth_arr, seq_vt, exclude_edge=EXCLUDE_EDGE, by_class=True)
        print(key)
        display(FLUX_SEQ[key].round(4))
CUM = pd.concat([fx.cumulative(PARTS[FINAL], H.truth_arr, d, exclude_edge=EXCLUDE_EDGE) for d in seq_vt], ignore_index=True)
for key in ('OOD', FINAL):
    P = PARTS[key][PARTS[key].seq_id.str.endswith('test')]
    print('carbon budget, test,', key)
    display(cl.carbon_budget(P, P.carbon_pred.values, H.truth_arr, fx.counted_mask(P, EXCLUDE_EDGE)).round(2))
with pd.ExcelWriter(f'{EVAL}/stage5_flux.xlsx') as xw:
    for key in FLUX_SEQ:
        FLUX_SEQ[key].to_excel(xw, sheet_name=f'flux {key}'[:31], index=False)
        FLUX_CLS[key].to_excel(xw, sheet_name=f'by class {key}'[:31], index=False)
        fx.mae_by_class(FLUX_CLS[key]).to_excel(xw, sheet_name=f'MAE {key}'[:31])
    CUM.to_excel(xw, sheet_name='cumulative', index=False)
fg.cumulative_carbon(CUM, f'{FIGS}/stage5_cumulative_carbon.png'); plt.show()

## Part G: figures

In [ ]:
# figures in the style of Song et al. (2025) and Amaral & Durkin (2025)
fg.preprocessing(TEST_SEQ, FRAMES // 2, OUT, f'{FIGS}/song_fig2'); plt.show()
Tv, Cv = H.tables('test', crops=True)
fg.class_examples(Tv[Tv.uid > 0], Cv[(Tv.uid > 0).values], list(fg.CLASS_COLORS)[:8],
                  out_png=f'{FIGS}/song_fig4_class_examples.png'); plt.show()
for d in seqs(DATA, ('test',)):
    sid = os.path.basename(d)
    fg.cumulative_area(CUM, fx.white_pixels(d), sid, f'{FIGS}/song_fig5_cumulative_area_{sid}.png'); plt.show()
fg.flux_by_type(FLUX_CLS[FINAL], f'{FIGS}/song_fig6_flux_by_type.png', f'POC flux by particle type ({FINAL})'); plt.show()
fg.flux_scatter(FLUX_CLS, f'{FIGS}/amaral_fig5_flux_vs_truth.png'); plt.show()
fg.mae_by_stage({k: fx.mae_by_class(v) for k, v in FLUX_CLS.items()}, f'{FIGS}/amaral_fig6_mae.png'); plt.show()
fg.precision_recall({k: v for k, v in pr.items() if k.startswith('test')}, f'{FIGS}/amaral_fig7_precision_recall.png'); plt.show()
fg.flux_by_class_stage(FLUX_CLS, f'{FIGS}/amaral_fig8_flux_by_class.png'); plt.show()

## Part H: summary

In [ ]:
# every stage's results in one workbook
read = lambda f: pd.read_csv(f'{EVAL}/{f}') if os.path.exists(f'{EVAL}/{f}') else None
trk_all = tk.summarise(pd.concat([pd.read_csv(f) for f in glob.glob(f'{OUT}/tracks/*_counts.csv') if os.path.getsize(f) > 2]), None)
S = ev.summary_table(read('stage2_segmentation.csv'), trk_all, read('stage5_carbon_errors.csv'), FLUX_SEQ.get(FINAL))
display(S)
with pd.ExcelWriter(f'{OUT}/evaluation_vs_truth.xlsx') as xw:
    S.to_excel(xw, sheet_name='summary', index=False)
    for f, name in (('stage0_generator.csv', '0 generator'), ('stage1_preprocessing.csv', '1 preprocessing'),
                    ('stage2_segmentation.csv', '2 segmentation'), ('stage2_tau_curve.csv', '2 tau curve'),
                    ('stage3_tracking.csv', '3 tracking'), ('stage4_classification.csv', '4 classification'),
                    ('stage4_recall_per_class.csv', '4 recall per class'), ('stage5_carbon_errors.csv', '5 carbon')):
        if read(f) is not None:
            read(f).to_excel(xw, sheet_name=name, index=False)
    for key in FLUX_SEQ:
        FLUX_SEQ[key].to_excel(xw, sheet_name=f'5 flux {key}'[:31], index=False)
print('saved', f'{OUT}/evaluation_vs_truth.xlsx')